# Testing the RWA codes

In this notebook we test the RWA codes.<br>
I will compare the phase-space based computations to the one already implement in the monomial matrix paradigm.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import RWA_Lind as rwa
import time
import sympy as sp



In [ ]:
sD, sd, sg3, swp, swd= sp.symbols('\\Delta',positive=True), sp.symbols('\\delta',positive=True), sp.symbols('g_3',positive=True), sp.symbols('\\omega_p',positive=True), sp.symbols('\\omega_d',positive=True)
sr, sg1, sg2 = sp.symbols('r',positive=True), sp.symbols('g_1',positive=True), sp.symbols('g_2',positive=True)
sOd = sp.symbols('\\Omega_d',positive=True)
se_p = sp.symbols('\\epsilon_p^0',positive=True)
st = sp.symbols('t',positive=True)
se = sp.symbols('e',positive=True)
sedp = sp.exp(sp.I*(swp+swd)/2*st)
sed = sp.exp(sp.I*swd*st)
sep = sp.exp(sp.I*swp*st)

In [ ]:
# Old RWA scheme
S= rwa.System(5,2)# up to a^5b^5
S.make_structure_cste_ho()
S.make_ops()
zeros = np.zeros(S.H_0.shape,dtype=object)
Id = S.ladder((0,0,0,0))
a = S.ladder((0,1,0,0))
ad = S.ladder((1,0,0,0))
b = S.ladder((0,0,0,1))
bd = S.ladder((0,0,1,0))
He = (a*sedp.conjugate() + ad * sedp + sr*b*sed.conjugate() + sr * bd * sed)
sB = 1 #sp.symbols('B(t)')#function du bain
Hsb = sr *( b * sed.conjugate() + bd *sed) + (a*sedp.conjugate() + ad * sedp)
#Takes a few seconds
Ht = ((-sp.I)*sg3 * sp.besselj(1,se_p)*(sep-sep.conjugate()) *S.prod_custom(S.prod_custom(He,He),He) 
      +(-sp.I)*sg1 * sp.besselj(1,se_p)*(sep-sep.conjugate()) * He)
#      + sg2 * sp.besselj(0,se_p)*(sep+sep.conjugate()) *(S.prod_custom(He,He)))
Ht[1,1,0,0] += sD # a^dagger a
Ht[0,0,1,1] += sd
Ht[0,0,1,0] += 0*sp.I * sOd/2 
Ht[0,0,0,1] += -0*sp.I * sOd/2
#expanding and power simplification of the expressions, this is the longest operation of this cell
S.make_Ht(Ht,tim = st)
S.make_Hsb(Hsb)

#S.RWA_2()

In [ ]:
S.RWA_1()

In [ ]:
E = rwa.Expression()
E.make_ladders(2)
He = (E.a[0] *sedp.conjugate() + E.ad[0] * sedp + sr* E.a[1] *sed.conjugate() + sr * E.ad[1] * sed)

In [ ]:
#New RWA scheme
E = rwa.Expression()
E.make_ladders(2)
He = (E.a[0] *sedp.conjugate() + E.ad[0] * sedp + sr* E.a[1] *sed.conjugate() + sr * E.ad[1] * sed)
Hsb = sr *( E.a[1] * sed.conjugate() + E.ad[1] *sed) + (E.a[0] *sedp.conjugate() + E.ad[0] * sedp)
Ht = ((-sp.I)*sg3 * sp.besselj(1,se_p)*(sep-sep.conjugate()) * E.moyal_product(E.moyal_product(He,He),He)
        +(-sp.I)*sg1 * sp.besselj(1,se_p)*(sep-sep.conjugate()) * He) + sd * E.ad[0] * E.a[0] + sD * E.ad[1] * E.a[1]
E.make_Ht(Ht,tim = st)
E.make_Hsb(Hsb)


In [ ]:
Ht

In [ ]:
E.RWA_1()
E.RWA_2()

In [ ]:
E.S2

In [ ]:
S.print_operator(S.Hb)

In [ ]:
E.JRWA(1)

In [ ]:
#Time the moyal_product function
start = time.time()
E.moyal_product(E.S1,E.Hsb)
end = time.time()
print(end - start)

In [ ]:
Hes = He
Hts = Ht

In [ ]:
#Time the prod_custom function
start = time.time()
S.prod_custom(S.S1,S.Hsb)
end = time.time()
print(end - start)

In [ ]:
#time the formating function
start = time.time()
#sp.powsimp(sp.expand(Ht,power_exp=False))
E.formating_rotating(Ht)
end = time.time()
print(end - start)


In [ ]:
#Time the S.RWA_1 function
start = time.time()
S.formating_rotating(Hts)
end = time.time()
print(end - start)

In [ ]:

O_copy = 0 * O
O_copy = sp.expand(O_copy,power_exp=False)
Expr2 = 0*sp.I
if type(O_copy)==sp.core.add.Add:
    for a in O_copy.args:
        Expr2 += sp.powsimp(sp.factor_terms(a),force=True)
    O_copy = Expr2
O_copy

In [ ]:
# Factorial of the list ([1,2,5]) using numpy
np.prod([1,2,5])

